In [ ]:
import os

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Google Drive mount skipped/failed:", exc)

# Edit these two paths if your Drive layout is different.
DRIVE_ROOT = "/content/drive/MyDrive"
CHECKPOINT_ROOT = "/content/drive/MyDrive/Anh Khôi ĐACN/Thực nghiệm/100 clients/NICE_IL"
DATA_DIR = "/content/drive/MyDrive/Anh Khôi ĐACN/Dataset/2023/federated_splits/100-clients"

pt_files = []
for root, _, files in os.walk(CHECKPOINT_ROOT):
    for f in files:
        if f.endswith(".pt"):
            pt_files.append(os.path.join(root, f))

print("PT files found:")
for p in sorted(pt_files):
    print(p)

print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("DATA_DIR exists:", os.path.exists(DATA_DIR))


In [ ]:
import os

print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)
print("exists:", os.path.exists(CHECKPOINT_ROOT))

if os.path.exists(CHECKPOINT_ROOT):
    print("\nTop-level contents:")
    for name in sorted(os.listdir(CHECKPOINT_ROOT)):
        path = os.path.join(CHECKPOINT_ROOT, name)
        kind = "DIR " if os.path.isdir(path) else "FILE"
        size = os.path.getsize(path) if os.path.isfile(path) else ""
        print(f"{kind} {name} {size}")

    print("\n.pt/.pth preview under CHECKPOINT_ROOT:")
    count = 0
    for root, _, files in os.walk(CHECKPOINT_ROOT):
        for filename in sorted(files):
            if filename.lower().endswith((".pt", ".pth")):
                print(os.path.join(root, filename))
                count += 1
                if count >= 80:
                    print("... truncated at 80 files")
                    break
        if count >= 80:
            break
    print("pt/pth shown:", count)


In [ ]:
import gc
import json
import os
import re
import shutil
import sys
from collections import OrderedDict

REPO_PATH = "/tmp/FL_IL_IDS"
CHECKPOINT_ROOT = globals().get("CHECKPOINT_ROOT", "/content/drive/MyDrive/Anh Khôi ĐACN/Thực nghiệm/100 clients/NICE_IL")
DATA_DIR = globals().get("DATA_DIR", "/content/drive/MyDrive/Anh Khôi ĐACN/Dataset/2023/federated_splits/100-clients")
OUTPUT_DIR = os.path.join(CHECKPOINT_ROOT, "eval_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def setup_imports():
    if os.path.exists(REPO_PATH):
        print(f"Removing stale clone at {REPO_PATH}...")
        shutil.rmtree(REPO_PATH)
    print("Cloning from GitHub...")
    os.system(f"git clone https://github.com/khoilv2005/FL_IL_IDS.git {REPO_PATH}")
    kaggle_prefix = "/kaggle/input"
    sys.path = [REPO_PATH] + [p for p in sys.path if not p.startswith(kaggle_prefix)]
    for name in list(sys.modules.keys()):
        if name == "fed_learning" or name.startswith("fed_learning."):
            del sys.modules[name]
    print("sys.path[0]:", sys.path[0])

setup_imports()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from eval_checkpoint import _make_model, _make_denice_client_model
from fed_learning.data.incremental_loader import IncrementalDataLoader
from fed_learning.training.local_task_loop import (
    _apply_local_nice_context_mask,
    _nice_seen_mask,
)
from fed_learning.training.denice_eval import (
    evaluate_denice_model,
    _denice_routed_logits_with_episodes,
)
from fed_learning.training.denice_delta_checkpoint import load_denice_checkpoint

def resolve_data_dir(data_dir):
    metadata_path = os.path.join(data_dir, "metadata.json")
    test_path = os.path.join(data_dir, "global_test_data.npz")
    if os.path.exists(metadata_path) and os.path.exists(test_path):
        return data_dir

    print("DATA_DIR invalid or incomplete:", data_dir)
    print("  metadata exists:", os.path.exists(metadata_path))
    print("  global_test_data exists:", os.path.exists(test_path))
    print("Searching Drive for folders containing metadata.json + global_test_data.npz...")

    search_roots = [
        globals().get("DRIVE_ROOT", "/content/drive/MyDrive"),
        "/content/drive/MyDrive",
    ]
    candidates = []
    for root in search_roots:
        if not root or not os.path.exists(root):
            continue
        for dirpath, _, files in os.walk(root):
            file_set = set(files)
            if "metadata.json" in file_set and "global_test_data.npz" in file_set:
                candidates.append(dirpath)
    candidates = sorted(set(candidates))
    print("Data dir candidates:")
    for c in candidates[:30]:
        print("  ", c)
    if not candidates:
        raise FileNotFoundError(
            "No folder with metadata.json and global_test_data.npz found. "
            "Set DATA_DIR to the exact federated split folder."
        )

    preferred = [c for c in candidates if "100" in c and "client" in c.lower()]
    selected = preferred[0] if preferred else candidates[0]
    print("Selected DATA_DIR:", selected)
    return selected

def list_round_checkpoints(root):
    if not os.path.exists(root):
        raise FileNotFoundError(f"CHECKPOINT_ROOT does not exist: {root}")

    pattern = re.compile(r"checkpoint_task_(\d+)_round_(\d+)\.pt$")
    checkpoints = []
    for dirpath, _, files in os.walk(root):
        for filename in files:
            m = pattern.match(filename)
            if not m:
                continue
            path = os.path.join(dirpath, filename)
            checkpoints.append(
                {
                    "task_id": int(m.group(1)),
                    "round_id": int(m.group(2)),
                    "path": path,
                    "mtime": os.path.getmtime(path),
                }
            )
    checkpoints = sorted(checkpoints, key=lambda r: (r["task_id"], r["round_id"], r["path"]))
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoint_task_<task>_round_<round>.pt found under {root}")
    return checkpoints

def _build_torch_context_router(context_detector, device):
    if context_detector is None or not getattr(context_detector, "context_learners", None):
        return None
    router = []
    for k, clf in enumerate(context_detector.context_learners):
        if clf is None or not hasattr(clf, "coef_") or not hasattr(clf, "intercept_"):
            router.append(None)
            continue
        coef = torch.as_tensor(clf.coef_[0], dtype=torch.float32, device=device)
        intercept = torch.as_tensor(float(clf.intercept_[0]), dtype=torch.float32, device=device)
        mask_np = context_detector.context_masks.get(k)
        if mask_np is None or not np.asarray(mask_np).any():
            mask_np = np.ones(int(coef.numel()), dtype=bool)
        mask_np = np.asarray(mask_np).astype(bool)
        if int(mask_np.sum()) != int(coef.numel()):
            # Fallback for old/incomplete checkpoints where mask dimensions drift.
            mask_np = np.ones(int(coef.numel()), dtype=bool)
        mask = torch.as_tensor(mask_np, dtype=torch.bool, device=device)
        router.append({"coef": coef, "intercept": intercept, "mask": mask})
    return router


def _binarize_context_activations_torch(context_detector, context_activations, device):
    parts = []
    thresholds = getattr(context_detector, "binarize_thresholds", None)
    for name in ["conv1", "conv2", "conv3", "gru"]:
        act = context_activations[name].detach().to(device)
        if thresholds is not None and name in thresholds:
            threshold = torch.as_tensor(float(thresholds[name]), dtype=act.dtype, device=device)
            binary = (act > threshold).float()
        else:
            binary = (act > 0).float()
        parts.append(binary)
    return torch.cat(parts, dim=1)


def _predict_episodes_torch(context_detector, router, binary_acts):
    latest_episode = max(context_detector.episode_classes.keys()) if context_detector.episode_classes else 0
    if not router:
        return torch.full((binary_acts.shape[0],), latest_episode, dtype=torch.long, device=binary_acts.device)

    pos_probs = []
    for entry in router:
        if entry is None:
            pos_probs.append(torch.zeros(binary_acts.shape[0], dtype=torch.float32, device=binary_acts.device))
            continue
        mask = entry["mask"]
        if mask.numel() == binary_acts.shape[1]:
            x = binary_acts[:, mask]
        else:
            x = binary_acts[:, : entry["coef"].numel()]
        logits = x.matmul(entry["coef"]) + entry["intercept"]
        pos_probs.append(torch.sigmoid(logits))

    pos = torch.stack(pos_probs, dim=1)
    neg = 1.0 - pos
    chain = torch.zeros(
        (binary_acts.shape[0], len(router) + 1),
        dtype=torch.float32,
        device=binary_acts.device,
    )
    for episode_index in range(len(router)):
        if episode_index == 0:
            chain[:, 0] = pos[:, 0]
        else:
            chain[:, episode_index] = torch.prod(neg[:, :episode_index], dim=1) * pos[:, episode_index]
    chain[:, -1] = torch.clamp(1.0 - chain.sum(dim=1), min=0.0)
    return torch.argmax(chain, dim=1)


def _apply_nice_context_mask_gpu(model, logits, context_detector, seen_classes, device, context_activations, router):
    if router is None or context_activations is None:
        return None
    if not getattr(context_detector, "episode_classes", None):
        return None

    binary_acts = _binarize_context_activations_torch(context_detector, context_activations, device)
    pred_episodes = _predict_episodes_torch(context_detector, router, binary_acts)
    masked = logits.clone()
    num_classes = masked.shape[1]
    seen_set = {int(c) for c in seen_classes}

    for episode in torch.unique(pred_episodes).tolist():
        allowed = [
            int(c)
            for c in context_detector.episode_classes.get(int(episode), [])
            if int(c) in seen_set and 0 <= int(c) < num_classes
        ]
        if not allowed:
            allowed = sorted(c for c in seen_set if 0 <= c < num_classes)
        if allowed:
            rows = pred_episodes == int(episode)
            row_idx = torch.nonzero(rows, as_tuple=False).flatten()
            if row_idx.numel() > 0:
                class_idx = torch.as_tensor(allowed, dtype=torch.long, device=device)
                masked[row_idx[:, None], class_idx] += 99999.0
    return masked

def evaluate_one_checkpoint(checkpoint_path, data_loader, test_cache, device, eval_batch_size):
    # Use load_denice_checkpoint to handle both delta and non-delta checkpoints
    ckpt = load_denice_checkpoint(checkpoint_path)
    config = dict(ckpt["config"])
    config["data_dir"] = DATA_DIR
    ckpt["config"] = config

    task_id = int(ckpt.get("task_id", config.get("task_end", 0)))
    round_id = int(ckpt.get("round_id", ckpt.get("final_round_id", -1)))
    algorithm = str(ckpt.get("algorithm", config.get("algorithm", ""))).lower()
    num_classes = int(config.get("total_classes", config.get("num_classes", 34)))

    if task_id not in test_cache:
        test_cache[task_id] = data_loader.get_test_data(task_id, cumulative=True)
    test_X, test_y = test_cache[task_id]
    if len(test_y) == 0:
        raise ValueError(
            f"No test samples for task {task_id}. Check DATA_DIR={DATA_DIR} and global_test_data.npz"
        )

    seen_classes = ckpt.get("seen_classes") or list(range(num_classes))
    labels = list(range(num_classes))
    stored_metrics = ckpt.get("metrics", {}) or {}

    # =================================================================
    # DeNICE (decentralized) path: per-client models
    # =================================================================
    is_denice = (
        algorithm == "denice"
        and "client_model_states" in ckpt
        and "model_state_dict" not in ckpt
    )
    if is_denice:
        client_ids = [
            int(cid)
            for cid in ckpt.get("client_ids", ckpt["client_model_states"].keys())
        ]
        per_client_metrics = []
        for cid in client_ids:
            model, context_detector = _make_denice_client_model(ckpt, cid, device)
            client_m = evaluate_denice_model(
                model,
                {"X_test": test_X, "y_test": test_y},
                device,
                context_detector=context_detector,
                seen_classes=seen_classes,
                batch_size=eval_batch_size,
            )
            per_client_metrics.append(client_m)
            del model, context_detector
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # Aggregate metrics across clients (mean)
        metric_keys = ["loss", "accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted"]
        mean_m = {
            k: sum(float(row[k]) for row in per_client_metrics) / max(1, len(per_client_metrics))
            for k in metric_keys
        }
        metrics = OrderedDict(
            checkpoint=checkpoint_path,
            algorithm=algorithm,
            task_id=task_id,
            round_id=round_id,
            train_loss=stored_metrics.get("train_loss"),
            test_loss=mean_m["loss"],
            accuracy=mean_m["accuracy"],
            precision_macro=mean_m["precision_macro"],
            recall_macro=mean_m["recall_macro"],
            f1_macro=mean_m["f1_macro"],
            f1_weighted=mean_m["f1_weighted"],
            eval_client_count=len(per_client_metrics),
        )

        # For confusion matrix: pick the best-accuracy client as representative
        best_idx = max(range(len(per_client_metrics)), key=lambda i: per_client_metrics[i]["accuracy"])
        best_cid = client_ids[best_idx]
        best_model, best_cd = _make_denice_client_model(ckpt, best_cid, device)
        best_model.eval()
        y_pred_list = []
        y_true_list = []
        with torch.no_grad():
            for start in range(0, len(test_y), eval_batch_size):
                xb = test_X[start:start + eval_batch_size].to(device)
                yb = test_y[start:start + eval_batch_size]
                logits, _ = _denice_routed_logits_with_episodes(
                    best_model, xb, best_cd, seen_classes, device,
                )
                y_pred_list.append(logits.argmax(dim=1).cpu().numpy())
                y_true_list.append(yb.numpy())
        y_true = np.concatenate(y_true_list)
        y_pred = np.concatenate(y_pred_list)
        del best_model, best_cd
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return metrics, y_true, y_pred, labels

    # =================================================================
    # Centralized path (NICE / DER / baseline)
    # =================================================================
    model, context_detector = _make_model(ckpt, device)
    torch_router = _build_torch_context_router(context_detector, device)
    criterion = nn.CrossEntropyLoss(reduction="sum")

    all_preds = []
    all_targets = []
    total_loss = 0.0
    model.eval()
    with torch.no_grad():
        for start in range(0, len(test_y), eval_batch_size):
            X_batch = test_X[start:start + eval_batch_size].to(device, non_blocking=True)
            y_batch = test_y[start:start + eval_batch_size].to(device, non_blocking=True)
            if (
                context_detector is not None
                and seen_classes is not None
                and hasattr(model, "get_output_and_context_activations")
            ):
                logits, context_activations = model.get_output_and_context_activations(X_batch)
            else:
                logits = model(X_batch)
                context_activations = None

            loss_logits = logits.clone()
            if context_detector is not None and seen_classes is not None:
                global_unseen = _nice_seen_mask(model, seen_classes, device)
                if len(global_unseen) == loss_logits.shape[1]:
                    loss_logits[:, global_unseen] = float("-inf")
                pred_logits = None
                try:
                    pred_logits = _apply_nice_context_mask_gpu(
                        model,
                        logits,
                        context_detector,
                        seen_classes,
                        device,
                        context_activations,
                        torch_router,
                    )
                except Exception as exc:
                    pred_logits = None
                if pred_logits is None:
                    pred_logits = _apply_local_nice_context_mask(
                        model,
                        logits,
                        X_batch,
                        context_detector,
                        seen_classes,
                        device,
                        context_activations=context_activations,
                    )
            else:
                pred_logits = logits

            total_loss += criterion(loss_logits, y_batch).item()
            all_preds.append(pred_logits.argmax(dim=1).detach().cpu().numpy())
            all_targets.append(y_batch.detach().cpu().numpy())
            del X_batch, y_batch, logits, pred_logits, loss_logits

    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_preds)
    metrics = OrderedDict(
        checkpoint=checkpoint_path,
        algorithm=algorithm,
        task_id=task_id,
        round_id=round_id,
        train_loss=stored_metrics.get("train_loss"),
        test_loss=total_loss / max(1, len(y_true)),
        accuracy=accuracy_score(y_true, y_pred),
        precision_macro=precision_score(y_true, y_pred, average="macro", labels=labels, zero_division=0),
        recall_macro=recall_score(y_true, y_pred, average="macro", labels=labels, zero_division=0),
        f1_macro=f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0),
        f1_weighted=f1_score(y_true, y_pred, average="weighted", labels=labels, zero_division=0),
    )
    return metrics, y_true, y_pred, labels

DATA_DIR = resolve_data_dir(DATA_DIR)
checkpoints = list_round_checkpoints(CHECKPOINT_ROOT)
print(f"Found {len(checkpoints)} round checkpoints")
print("First:", checkpoints[0]["path"])
print("Last :", checkpoints[-1]["path"])

device = "cuda" if torch.cuda.is_available() else "cpu"
eval_batch_size = 32768
print("device:", device)
print("eval_batch_size:", eval_batch_size)

data_loader = IncrementalDataLoader(data_dir=DATA_DIR)
test_cache = {}
all_rows = []
final_payload = None

for idx, item in enumerate(checkpoints, start=1):
    metrics, y_true, y_pred, labels = evaluate_one_checkpoint(
        item["path"], data_loader, test_cache, device, eval_batch_size
    )
    all_rows.append(metrics)
    final_payload = (metrics, y_true, y_pred, labels)
    print(
        f"[{idx}/{len(checkpoints)}] "
        f"task={metrics['task_id']} round={metrics['round_id']} "
        f"acc={metrics['accuracy'] * 100:.2f}% "
        f"f1w={metrics['f1_weighted'] * 100:.2f}%"
    )
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

all_json = os.path.join(OUTPUT_DIR, "all_round_metrics.json")
all_csv = os.path.join(OUTPUT_DIR, "all_round_metrics.csv")
with open(all_json, "w") as f:
    json.dump(all_rows, f, indent=2, default=str)
pd.DataFrame(all_rows).to_csv(all_csv, index=False)

# Backward-compatible final metrics files: last checkpoint in sorted task/round order.
final_metrics, final_y_true, final_y_pred, final_labels = final_payload
final_json = os.path.join(OUTPUT_DIR, "final_round_metrics.json")
final_csv = os.path.join(OUTPUT_DIR, "final_round_metrics.csv")
with open(final_json, "w") as f:
    json.dump(final_metrics, f, indent=2, default=str)
pd.DataFrame([final_metrics]).to_csv(final_csv, index=False)

cm = confusion_matrix(final_y_true, final_y_pred, labels=final_labels)
cm_csv_path = os.path.join(OUTPUT_DIR, "confusion_matrix_34.csv")
cm_png_path = os.path.join(OUTPUT_DIR, "confusion_matrix_34.png")
cm_pdf_path = os.path.join(OUTPUT_DIR, "confusion_matrix_34.pdf")
pd.DataFrame(cm, index=final_labels, columns=final_labels).to_csv(cm_csv_path)

plt.figure(figsize=(24, 20))
sns.heatmap(cm, cmap="Blues", xticklabels=final_labels, yticklabels=final_labels, cbar=True)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title(
    f"Confusion Matrix - {len(final_labels)} classes - "
    f"task {final_metrics['task_id']} round {final_metrics['round_id']}"
)
plt.tight_layout()
plt.savefig(cm_png_path, dpi=200)
plt.savefig(cm_pdf_path)
plt.show()

print("Saved all-round metrics JSON:", all_json)
print("Saved all-round metrics CSV:", all_csv)
print("Saved final metrics JSON:", final_json)
print("Saved final metrics CSV:", final_csv)
print("Saved final confusion CSV:", cm_csv_path)
print("Saved final confusion PNG:", cm_png_path)
print("Saved final confusion PDF:", cm_pdf_path)
pd.DataFrame(all_rows).tail()


In [ ]:
import json
import os

import pandas as pd

# Scan all result folders and merge every final_round_metrics.json into one table.
METRICS_ROOTS = [
    globals().get("CHECKPOINT_ROOT", "/content/drive/MyDrive"),
    globals().get("OUTPUT_DIR", os.path.join(globals().get("CHECKPOINT_ROOT", "/content/drive/MyDrive"), "eval_outputs")),
]
MERGED_OUTPUT_DIR = globals().get("OUTPUT_DIR", os.path.join(globals().get("CHECKPOINT_ROOT", "/content/drive/MyDrive"), "eval_outputs"))
os.makedirs(MERGED_OUTPUT_DIR, exist_ok=True)

rows = []
seen_paths = set()
for root_dir in METRICS_ROOTS:
    if not root_dir or not os.path.exists(root_dir):
        continue
    for root, _, files in os.walk(root_dir):
        if "final_round_metrics.json" not in files:
            continue
        path = os.path.join(root, "final_round_metrics.json")
        if path in seen_paths:
            continue
        seen_paths.add(path)
        try:
            with open(path, "r") as f:
                record = json.load(f)
            record["metrics_file"] = path
            record["run_dir"] = root
            rows.append(record)
        except Exception as exc:
            print("Failed to load:", path, exc)

rows = sorted(
    rows,
    key=lambda r: (
        str(r.get("algorithm", "")),
        int(r.get("task_id", -1) or -1),
        int(r.get("round_id", -1) or -1),
        str(r.get("checkpoint", "")),
    ),
)

merged_json = os.path.join(MERGED_OUTPUT_DIR, "all_final_round_metrics.json")
merged_csv = os.path.join(MERGED_OUTPUT_DIR, "all_final_round_metrics.csv")
with open(merged_json, "w") as f:
    json.dump(rows, f, indent=2, default=str)
pd.DataFrame(rows).to_csv(merged_csv, index=False)

print(f"Merged {len(rows)} final_round_metrics.json files")
print("Saved merged JSON:", merged_json)
print("Saved merged CSV:", merged_csv)
pd.DataFrame(rows).head()
